# Data analysis

Exploration of the raw sources (raw) before cleansing.
Findings go in [ANALYSIS.md](../ANALYSIS.md).

Sections:
1. Setup
2. Load raw sources
3. Generic profiling
4. Products analysis
5. Customers analysis
6. Orders analysis
7. Cross-source integrity




## 1. Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F


spark = (
    SparkSession.builder
    .appName("data_analysis")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:

RAW = "../data/raw"
SOURCES = {
    "products": f"{RAW}/products.csv",
    "customers": f"{RAW}/customers.csv",
    "orders": f"{RAW}/orders.csv",
}

## 2. Load raw sources

In [ ]:
raw = {
    name: spark.read.option("header", True).csv(path)
    for name, path in SOURCES.items()
}

## 3. Generic profiling

In [ ]:
def profile(df, name):
    """Row count, schema, nulls/blanks per column and duplicate rows."""
    total = df.count()
    print(f"==> {name}: {total:,} rows, {len(df.columns)} columns")
    df.printSchema()

    null_or_blank = [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    df.select(null_or_blank).show(truncate=False)

    print(f"exact duplicate rows: {total - df.distinct().count():,}")


for name, df in raw.items():
    profile(df, name)

## 4. Products analysis

Ingested daily as snapshots: the same `StockCode` is expected to appear on
several days. Check the snapshot date column and the price distribution.

### 4.1 Volume and key uniqueness

In [ ]:
products = raw["products"]
print(
    "rows:", products.count(),
    "| distinct StockCode:",
    products.select("StockCode").distinct().count()
)

### 4.2 Price quality

In [ ]:
products.select(
    F.min(F.col("UnitPrice").cast("double")).alias("min_price"),
    F.max(F.col("UnitPrice").cast("double")).alias("max_price"),
    F.sum(
        F.when(F.col("UnitPrice").cast("double") <= 0, 1).otherwise(0)
    ).alias("non_positive_prices"),
).show()

### 4.3 Repeated StockCode investigation

In [ ]:
products.groupBy("StockCode").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Description").alias("descriptions"),
    F.countDistinct("UnitPrice").alias("prices")
).filter(
    F.col("rows") > 1
).orderBy(
    F.desc("rows")
).show(10)

In [ ]:
products.filter(
    F.col("StockCode") == "23084"
).show(truncate=False)

In [ ]:
products.withColumn(
    "is_uppercase",
    F.col("Description") == F.upper(F.col("Description"))
).groupBy("is_uppercase").count().show()


products.filter(
    F.col("Description").isNotNull() &
    (F.col("Description") != F.upper(F.col("Description")))
).select(
    "StockCode", "Description", "UnitPrice"
).show(100, truncate=False)



In [ ]:
non_upper = products.filter(
    F.col("Description").isNotNull() &
    (F.col("Description") != F.upper(F.col("Description")))
)

print("Non-uppercase descriptions:", non_upper.count())

non_upper.groupBy("Description") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

In [ ]:
product_case_flags = products.groupBy("StockCode").agg(
    F.max(
        F.when(
            F.col("Description") == F.upper(F.col("Description")),
            1
        ).otherwise(0)
    ).alias("has_upper"),

    F.max(
        F.when(
            F.col("Description") != F.upper(F.col("Description")),
            1
        ).otherwise(0)
    ).alias("has_non_upper")
)

product_case_flags.filter(
    (F.col("has_upper") == 1) &
    (F.col("has_non_upper") == 1)
).count()

In [ ]:
product_case_flags.filter(
    (F.col("has_upper") == 0) &
    (F.col("has_non_upper") == 1)
).show(20, truncate=False)

In [ ]:
non_upper_only_codes = (
    product_case_flags
    .filter(
        (F.col("has_upper") == 0) &
        (F.col("has_non_upper") == 1)
    )
    .select("StockCode")
)

products.join(
    non_upper_only_codes,
    "StockCode",
    "inner"
).orderBy("StockCode").show(20, truncate=False)

## 5. Customers analysis

### 5.1 Volume and key uniqueness

In [ ]:
customers = raw["customers"]
print("distinct CustomerID:", customers.select("CustomerID").distinct().count())

customers.groupBy("Country").count().orderBy(F.desc("count")).show(10, truncate=False)

### 5.2 Conflicting customer-country mappings

In [ ]:
customers.groupBy("CustomerID").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Country").alias("countries")
).filter(
    F.col("rows") > 1
).orderBy(F.desc("rows")).show(20, truncate=False)

In [ ]:
customers.filter(
    F.col("CustomerID").isin(
        "12394", "12429", "12457", "12431",
        "12417", "12370", "12455", "12422"
    )
).orderBy("CustomerID").show(truncate=False)

## 6. Orders analysis

Things to look at: invoices starting with `C` (cancellations?), negative
quantities.

### 6.1 Volume and cardinality

In [ ]:
orders = raw["orders"]
print("distinct InvoiceNo:", orders.select("InvoiceNo").distinct().count())

orders.select("InvoiceDate").show(5, truncate=False) 

In [ ]:
orders.select(
    F.countDistinct("InvoiceNo").alias("distinct_invoices"),
    F.countDistinct("StockCode").alias("distinct_stock_codes"),
    F.countDistinct("CustomerID").alias("distinct_customers")
).show()

### 6.2 Quantity analysis

In [ ]:
orders.select(
    F.min(F.col("Quantity").cast("int")).alias("min_quantity"),
    F.max(F.col("Quantity").cast("int")).alias("max_quantity"),
    F.sum(
        F.when(F.col("Quantity").cast("int") <= 0, 1).otherwise(0)
    ).alias("non_positive_quantity")
).show()

### 6.3 Cancellation / negative quantity investigation

In [ ]:
orders.select(
    F.sum(F.when(F.col("InvoiceNo").startswith("C"), 1).otherwise(0)).alias("cancellation_lines"),
    F.sum(F.when(F.col("Quantity").cast("int") < 0, 1).otherwise(0)).alias("negative_quantity_lines"),
).show()

In [ ]:
orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
).groupBy("StockCode").count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

In [ ]:
orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
).select(
    "InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "CustomerID"
).show(30, truncate=False)

In [ ]:
negative_non_cancelled = orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
)

negative_non_cancelled.groupBy(
    F.col("CustomerID").isNull().alias("customer_id_null")
).count().show()

### 6.4 Candidate grain investigation

In [ ]:
orders.groupBy("InvoiceNo", "StockCode").count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

In [ ]:
orders_dedup = orders.dropDuplicates()

duplicate_grain = (
    orders_dedup
    .groupBy("InvoiceNo", "StockCode")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Repeated (InvoiceNo, StockCode) after exact dedup:",
    duplicate_grain.count()
)

duplicate_grain.orderBy(F.desc("count")).show(20)

In [ ]:
orders.dropDuplicates().filter(
    (F.col("InvoiceNo") == "568188") &
    (F.col("StockCode") == "84879")
).show(truncate=False)


## 7. Cross-source integrity

### 7.1 Customer integrity

In [ ]:
orphan_customers = orders.join(customers, "CustomerID", "left_anti")
print("order lines with unknown CustomerID:", orphan_customers.count())

In [ ]:
# Unknown non-null customers
orders.filter(F.col("CustomerID").isNotNull()) \
    .join(
        customers.select("CustomerID").distinct(),
        "CustomerID",
        "left_anti"
    ).count()




### 7.2 Product integrity

In [ ]:
orphan_products = orders.join(products, "StockCode", "left_anti")
print("order lines with unknown StockCode:", orphan_products.count())

orphan_products.select("StockCode").distinct().show(10, truncate=False)

In [ ]:
# Which StockCodes are missing from products?
orders.select("StockCode").distinct() \
    .join(
        products.select("StockCode").distinct(),
        "StockCode",
        "left_anti"
    ).show(20, truncate=False)

### 7.3 StockCode normalization

In [ ]:
orders_stockcodes = orders.select(
    F.upper(F.trim("StockCode")).alias("StockCode")
).distinct()

products_stockcodes = products.select(
    F.upper(F.trim("StockCode")).alias("StockCode")
).distinct()

unknown_stockcodes = orders_stockcodes.join(
    products_stockcodes,
    "StockCode",
    "left_anti"
)

print("Unknown StockCodes after normalization:", unknown_stockcodes.count())
unknown_stockcodes.show(100, truncate=False)

## Analysis for data model

In [ ]:
# Analyze how many distinct prices exist for each product
products.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy("distinct_prices").show()

In [ ]:
# Measure the price range for each product to assess the magnitude of price variation
product_price_range = (
    products
    .withColumn("UnitPrice_num", F.col("UnitPrice").cast("double"))
    .groupBy("StockCode")
    .agg(
        F.countDistinct("UnitPrice_num").alias("distinct_prices"),
        F.min("UnitPrice_num").alias("min_price"),
        F.max("UnitPrice_num").alias("max_price")
    )
    .withColumn(
        "price_range",
        F.col("max_price") - F.col("min_price")
    )
)

product_price_range.filter(
    F.col("distinct_prices") > 1
).orderBy(F.desc("price_range")).show(30, truncate=False)

In [ ]:
# Inspect all product records for StockCodes that have exactly two distinct prices
products_with_two_prices = (
    product_price_range
    .filter(F.col("distinct_prices") == 2)
    .select("StockCode")
)

products.join(
    products_with_two_prices,
    on="StockCode",
    how="inner"
).orderBy("StockCode", "UnitPrice").show(100, truncate=False)

In [ ]:
# Check product price uniqueness after excluding records with a missing description
products_with_description = products.filter(
    F.col("Description").isNotNull()
)

products_with_description.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy("distinct_prices").show()

In [ ]:
# Inspect products that still have exactly two distinct prices after removing null descriptions
two_price_codes = (
    products_with_description
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .filter(F.col("distinct_prices") == 2)
    .select("StockCode")
)

products_with_description.join(
    two_price_codes,
    on="StockCode",
    how="inner"
).orderBy("StockCode").show(100, truncate=False)

In [ ]:
# Identify records whose full description exactly matches known operational terms
technical_terms = [
    "check",
    "damaged",
    "found",
    "adjustment",
    "amazon",
    "dotcom",
    "wet damaged",
    "wet/rusty",
    "stock check"
]

technical_records = products.filter(
    F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

technical_records.groupBy("Description").count().orderBy(
    F.desc("count")
).show(100, truncate=False)


# Measure remaining price ambiguity after removing null and known operational descriptions
clean_products_candidate = products.filter(
    F.col("Description").isNotNull() &
    ~F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

clean_products_candidate.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy(
    "distinct_prices"
).show()

In [ ]:
# Measure price uniqueness after removing null and known operational descriptions
clean_products_candidate = products.filter(
    F.col("Description").isNotNull() &
    ~F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

price_distribution = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .groupBy("distinct_prices")
    .count()
    .orderBy("distinct_prices")
)

price_distribution.show()

In [ ]:
# Measure the impact on order lines if products with multiple prices are excluded
ambiguous_products = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .filter(F.col("distinct_prices") > 1)
    .select("StockCode")
)

orders_normalized = orders.withColumn(
    "StockCode_normalized",
    F.upper(F.trim(F.col("StockCode")))
)

ambiguous_products_normalized = ambiguous_products.withColumn(
    "StockCode_normalized",
    F.upper(F.trim(F.col("StockCode")))
)

affected_product_orders = (
    orders_normalized
    .join(
        ambiguous_products_normalized.select("StockCode_normalized"),
        "StockCode_normalized",
        "inner"
    )
)

print("Affected order lines:", affected_product_orders.count())
print("Total order lines:", orders.count())
print(
    "Percentage:",
    affected_product_orders.count() / orders.count() * 100
)

In [ ]:
# Measure the impact on order lines if customers associated with multiple countries are excluded
ambiguous_customers = (
    customers
    .filter(F.col("CustomerID").isNotNull())
    .groupBy("CustomerID")
    .agg(F.countDistinct("Country").alias("country_count"))
    .filter(F.col("country_count") > 1)
    .select("CustomerID")
)

affected_customer_orders = orders.join(
    ambiguous_customers,
    "CustomerID",
    "inner"
)

print("Affected order lines:", affected_customer_orders.count())
print("Total order lines:", orders.count())
print(
    "Percentage:",
    affected_customer_orders.count() / orders.count() * 100
)

In [ ]:
# Compare simple deterministic price-selection strategies for ambiguous products
price_strategies = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(
        F.min(F.col("UnitPrice").cast("double")).alias("min_price"),
        F.max(F.col("UnitPrice").cast("double")).alias("max_price"),
        F.avg(F.col("UnitPrice").cast("double")).alias("avg_price"),
        F.countDistinct("UnitPrice").alias("price_count")
    )
    .filter(F.col("price_count") > 1)
)

price_strategies.orderBy(F.desc("price_count")).show(30, truncate=False)

In [ ]:
# Inspect ambiguous products to check whether the first source record looks like the canonical product record
products.filter(
    F.col("StockCode").isin(
        "23084",
        "20713",
        "23343",
        "21830",
        "23131",
        "85175"
    )
).show(100, truncate=False)